# 📘 Notebook 04: Learning from Complete Episodes

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module A: Foundations · Notebook 4 of 4 in this module · (4 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what changes when the agent does **not know the rules** of its environment
- Estimate the value of a policy by **Monte Carlo**: averaging the returns of complete episodes
- Explain why learning without a model needs **action values** $q(s, a)$ and not only state values
- Implement **Monte Carlo control** with an epsilon-greedy policy and learn to play Blackjack
- Read a learned policy as a table and compare it with a known strategy
- Name the strengths and the limits of Monte Carlo methods

> **Prerequisites:** Notebooks 01 to 03: the incremental average, returns, policies, and the value functions $v_\pi$ and $q_\pi$.
>
> 🔭 **Why this notebook matters later:** this is the first time in the course that an agent learns good behaviour from experience alone. The same loop (play with the current policy, update the estimates, act a little more greedily) drives the snake of Notebook 07 and the policy-gradient agents of Module D, which also learn from complete episodes.

> 📦 **Libraries used in this notebook.** **Gymnasium** provides the Blackjack environment, and **imageio** turns pictures of the table into a video. The cell below installs them. The cells of this notebook take a minute or two in total on an ordinary processor.

In [ ]:
%pip install -q "gymnasium[toy-text]>=1.3" imageio imageio-ffmpeg

In [ ]:
import base64
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
from IPython.display import HTML, display

random.seed(0)


def show_video(frames, fps=1, width=360):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. Learning without the rule book

### Intuition first
In Notebook 03 the robot was handed the complete rules of its room and computed the best behaviour without taking a step. You did not learn card games that way. Nobody gave you the probability of every possible deal. You played many hands, remembered roughly how things turned out, and your judgement improved.

### Formal definition
A method is **model-free** when it uses only **experience**: sequences of states, actions and rewards obtained by actually playing. It never looks at the transition probabilities $p(s' \mid s, a)$.

The simplest model-free idea carries the name of a casino. A **Monte Carlo** method estimates an expected value by **averaging samples**. The value of a state is the expected return from it, so:

> play many episodes, and for each state average the returns that followed it.

That is the whole method. Notebook 03 already used it as a check on the Bellman calculation. Here it is all we have.

## 2. The game: Blackjack

Blackjack suits Monte Carlo learning well. Episodes are short, the rules are simple to state, and the probabilities are awkward enough that nobody wants to write the model by hand.

**The rules, as implemented in Gymnasium:**

- The aim is a hand whose cards add up to more than the dealer's, without going over 21.
- Number cards count their value, picture cards count 10, and an ace counts 11 or 1, whichever is better. An ace currently counted as 11 is called a **usable ace**.
- The player starts with two cards and sees **one** of the dealer's two cards.
- The player may **hit** (take another card) as often as they like, or **stick** (stop). Going over 21 is a **bust** and loses at once.
- When the player sticks, the dealer draws cards until reaching 17 or more.
- The reward is +1 for a win, −1 for a loss and 0 for a draw, given at the end of the hand.

| | Meaning |
|---|---|
| State | `(player's total, dealer's visible card, usable ace?)`, for example `(14, 10, 0)` |
| Actions | `0` stick, `1` hit |

In [ ]:
env = gym.make("Blackjack-v1", sab=True)          # sab: the rules as in Sutton and Barto's textbook


def play_hand(env, policy):
    """Play one hand. Returns the episode as a list of (state, action, reward)."""
    episode = []
    state, info = env.reset()
    done = False
    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode.append((state, action, reward))
        state, done = next_state, terminated or truncated
    return episode


def random_policy(state):
    return random.randint(0, 1)


env.reset(seed=0)
for hand in range(4):
    episode = play_hand(env, random_policy)
    steps = ",  ".join(f"{state} {'hit' if action else 'stick'}" for state, action, reward in episode)
    print(f"{steps}   ->  reward {episode[-1][2]:+.0f}")

Every hand is one short **episode**, and the only non-zero reward comes at the very end. That delay is typical: the agent has to work out which of its earlier decisions deserve the credit or the blame.

Before learning anything, let us measure two simple policies. The second is a cautious player who only stops on 20 or 21.

In [ ]:
def stick_on_20(state):
    player_total, dealer_card, usable_ace = state
    return 0 if player_total >= 20 else 1


def average_reward(env, policy, hands=20000):
    return float(np.mean([play_hand(env, policy)[-1][2] for hand in range(hands)]))


print(f"random play:     average reward per hand {average_reward(env, random_policy):+.3f}")
print(f"stick on 20, 21: average reward per hand {average_reward(env, stick_on_20):+.3f}")

Both lose money on average, as expected against a casino. An average of −0.35 means losing 35 chips per 100 hands when betting one chip a hand. Remember these two numbers: the learning agent will have to beat them.

## 3. Monte Carlo prediction: how good is a policy?

### Formal definition
To estimate $v_\pi(s)$ for a fixed policy $\pi$:

1. Play an episode with $\pi$.
2. Walk **backwards** through it, computing the return $G$ that followed each step ($G \leftarrow R + \gamma G$, as in Notebook 02).
3. For every state visited, add that return to the state's running average.
4. Repeat for many episodes.

If a state appears twice in one episode, **first-visit** Monte Carlo uses only the return after its first appearance. (In Blackjack a state can never repeat within a hand, so the distinction does not arise.) Episodes are short and always end, so we use $\gamma = 1$.

In [ ]:
def mc_prediction(env, policy, episodes):
    """First-visit Monte Carlo estimate of the state values of a policy."""
    value = defaultdict(float)
    visits = defaultdict(int)
    for i in range(episodes):
        episode = play_hand(env, policy)
        states = [state for state, action, reward in episode]
        G = 0.0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = reward + G                                   # gamma = 1
            if state not in states[:t]:                      # first visit only
                visits[state] += 1
                value[state] += (G - value[state]) / visits[state]    # the incremental average of Notebook 01
    return value


V = mc_prediction(env, stick_on_20, episodes=100000)

for state in [(21, 10, 0), (20, 10, 0), (16, 10, 0), (13, 2, 0), (13, 2, 1)]:
    print(f"player {state[0]}, dealer shows {state[1]:>2}, usable ace {bool(state[2])!s:5}:  value {V[state]:+.2f}")

The numbers make sense to anyone who has played. Holding 21 is very good, holding 20 is good, and holding 16 against a dealer's 10 is bad, for this policy: it will keep hitting on 16 and usually bust.

A picture shows the whole value function at once.

In [ ]:
def show_table(table, title, cmap="RdYlGn", vmin=-1, vmax=1):
    """Draw one number per state: rows are the player's total, columns the dealer's card."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, usable in zip(axes, (0, 1)):
        grid = np.array([[table(total, dealer, usable) for dealer in range(1, 11)] for total in range(21, 11, -1)])
        image = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(10)); ax.set_xticklabels(["A"] + list(range(2, 11)))
        ax.set_yticks(range(10)); ax.set_yticklabels(range(21, 11, -1))
        ax.set_xlabel("dealer's visible card"); ax.set_ylabel("player's total")
        ax.set_title(f"{title}\n({'with' if usable else 'without'} a usable ace)")
    fig.colorbar(image, ax=axes, shrink=0.8)
    plt.show()


show_table(lambda total, dealer, usable: V[(total, dealer, usable)], "Value of 'stick on 20 or 21'")

Only the top two rows are green: this policy does well when it is dealt 20 or 21 and poorly almost everywhere else. The right-hand picture is noisier, because hands with a usable ace are rarer and their averages rest on fewer samples.

No probability was written down anywhere. The agent treated the game as a black box that deals cards and announces results.

## 4. From judging to improving: why we need action values

### Intuition first
In Notebook 03 a policy was improved by looking one step ahead: "which action leads to the best next state?" That question needs the model, to know where each action leads. A Blackjack player cannot answer it: they do not know what the next card will be.

The way out is to learn the value of each **action** directly. If I know from experience that "hitting on 16 against a 10" has averaged −0.5 and "sticking" has averaged −0.6, I can choose between them without knowing anything about the deck.

### Formal definition
Model-free control estimates $q(s, a)$, one number per **state-action pair**, in a table called the **Q-table**. Improving the policy is then a table lookup: in state $s$ choose $\arg\max_a Q(s, a)$.

### The exploration problem returns
There is a catch that Notebook 01 prepared us for. If the agent always takes the action that currently looks best, the other action in that state is never tried again and its estimate never improves. The cure is the same as for the slot machines: an **epsilon-greedy** policy. We start with a lot of exploration and reduce it as the estimates become trustworthy.

## 5. Monte Carlo control

The complete algorithm repeats three steps, hand after hand:

1. **Play** a hand with the epsilon-greedy policy for the current Q-table.
2. **Evaluate**: move $Q(s, a)$ of every pair that occurred a small step towards the return that followed it.
3. **Improve**: nothing to do. The policy reads the Q-table, so it has already changed.

Evaluation and improvement are interleaved, hand by hand, the way value iteration interleaved them sweep by sweep.

One detail differs from Section 3. The update uses a small **constant step size** `alpha` in place of `1 / visits`. An exact average gives the same weight to every return ever seen, including those from the first hands, when the agent was playing almost at random. Those returns describe a policy that no longer exists. A constant step lets old experience fade, as it did for the drifting slot machines in Exercise 6 of Notebook 01. Exercise 6 below measures the difference.

In [ ]:
def mc_control(env, episodes, alpha=0.01, epsilon_end=0.05):
    Q = defaultdict(lambda: np.zeros(2))                 # Q[state] = [value of stick, value of hit]
    for i in range(episodes):
        epsilon = max(epsilon_end, 1.0 - i / (0.6 * episodes))      # from 1 down to 0.05

        def policy(state):
            if random.random() < epsilon:
                return random.randint(0, 1)                         # explore
            return int(Q[state].argmax())                           # exploit

        episode = play_hand(env, policy)
        G = 0.0
        for state, action, reward in reversed(episode):
            G = reward + G
            Q[state][action] += alpha * (G - Q[state][action])      # a small step towards the return
    return Q


random.seed(0)
env.reset(seed=0)
Q = mc_control(env, episodes=300000)
print(f"The Q-table has {len(Q)} states. Two examples:")
for state in [(16, 10, 0), (13, 5, 0)]:
    print(f"  {state}: stick {Q[state][0]:+.2f}, hit {Q[state][1]:+.2f}")

In [ ]:
def learned_policy(state):
    return int(Q[state].argmax())


print(f"random play:      {average_reward(env, random_policy):+.3f}")
print(f"stick on 20, 21:  {average_reward(env, stick_on_20):+.3f}")
print(f"learned policy:   {average_reward(env, learned_policy, hands=100000):+.3f}")

The learned policy still loses a little on average. That is not a failure of the learning: the rules favour the casino, and no policy has a positive expected reward in this version of the game. What matters is the comparison. The agent, which was told nothing but the final result of each hand, loses several times less than the cautious rule we wrote by hand.

What did it learn? The policy is small enough to draw completely.

In [ ]:
show_table(lambda total, dealer, usable: Q[(total, dealer, usable)].argmax(),
           "Learned policy: green = hit, red = stick", vmin=0, vmax=1)

Read the left-hand picture, the hands without a usable ace:

- With a high total the agent **sticks**, and with a low total it **hits**.
- Where the boundary lies depends on the **dealer's card**. Against a strong card (7 or higher, or an ace) the agent keeps hitting up to a higher total. Against a weak card (2 to 6) it stops early and lets the dealer take the risk of going bust.

Players call this pattern **basic strategy**, and it is printed in books about the game. The agent found its outline from results alone. With a usable ace (right-hand picture) it hits on higher totals, because an ace that can fall back to 1 makes a bust impossible on the next card.

A few squares may look out of place, especially on the right. States that occur rarely are estimated from few hands, and where the two actions are almost equally good a small error flips the choice.

Finally, let us watch the agent play a few hands. Each picture is one decision point.

In [ ]:
table = gym.make("Blackjack-v1", sab=True, render_mode="rgb_array")
frames = []
for hand in range(6):
    state, info = table.reset(seed=100 + hand)
    done = False
    while not done:
        frames.append(table.render())
        state, reward, terminated, truncated, info = table.step(learned_policy(state))
        done = terminated or truncated
    frames.append(table.render())
    frames.append(table.render())                 # hold the final position a little longer

show_video(frames)

> ⚠️ **Common pitfalls**
>
> - **Updating before the episode has ended.** A Monte Carlo method needs the complete return, so it can only learn when the hand is over.
> - **Trusting every entry of the table equally.** An estimate built from 20 samples is far less reliable than one built from 20,000. Keep the visit counts and look at them.
> - **Stopping exploration too early.** With no exploration the agent keeps the first habit that worked. Exercise 4 looks at how much that costs here.

## 6. What Monte Carlo methods can and cannot do

| Strength | Limit |
|---|---|
| Needs no model: experience is enough | Learns only at the **end** of an episode |
| Simple: an average of observed returns | Useless for tasks that never end |
| Each estimate is independent of the others, so an error in one state does not spread | The return of a long episode depends on many random events, so the estimates are **noisy** and learning is slow |

A hand of Blackjack lasts two or three decisions. A game of Snake can last thousands of steps, and waiting until the end before learning anything is wasteful: by then the agent has long forgotten which move mattered. The next notebook combines the sampling of Monte Carlo with the step-by-step reasoning of the Bellman equations, and learns **during** the episode.

---
## ✏️ Exercises

The exercises use `env`, `play_hand`, `average_reward`, `mc_prediction`, `mc_control` and `show_table` from above.

### Exercise 1 (Returns by hand)
An episode of some game visits the states A, B, C in that order and receives the rewards 2, 0 and 4. With gamma = 0.5, compute by hand the return that followed each state, then check with a short loop that works backwards.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
states, rewards, gamma = ["A", "B", "C"], [2, 0, 4], 0.5
G = 0.0
for state, reward in zip(reversed(states), reversed(rewards)):
    G = reward + gamma * G
    print(f"return after {state}: {G}")
```

*From C: 4. From B: 0 + 0.5 × 4 = 2. From A: 2 + 0.5 × 2 = 3. Working backwards computes all of them in one pass.*
</details>

### Exercise 2 (The dealer's own rule)
Write the policy that plays exactly as the dealer does: hit below 17, stick on 17 or more. Measure its average reward over 50,000 hands and compare it with `stick_on_20` and with the learned policy. If the player copies the dealer, why does the player still lose on average?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def like_the_dealer(state):
    return 0 if state[0] >= 17 else 1

print(f"stick on 20, 21:   {average_reward(env, stick_on_20, 50000):+.3f}")
print(f"like the dealer:   {average_reward(env, like_the_dealer, 50000):+.3f}")
print(f"learned policy:    {average_reward(env, learned_policy, 50000):+.3f}")
```

*The player moves first. If the player busts, the hand is lost at once, even when the dealer would have gone bust as well. That asymmetry is the casino's advantage. Notice also that this one-line rule is far better than sticking only on 20, and not very far behind the learned policy: a large part of what there is to gain in Blackjack comes from not hitting on high totals.*
</details>

### Exercise 3 (How many hands are enough?)
Estimate the value of the state `(13, 2, 0)` under `stick_on_20` using 1,000, 10,000 and 50,000 hands. Repeat each estimate four times with fresh hands and print the four numbers. What happens to their spread?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
state = (13, 2, 0)
for episodes in (1000, 10000, 50000):
    estimates = [mc_prediction(env, stick_on_20, episodes)[state] for repeat in range(4)]
    print(f"{episodes:>6} hands: " + "  ".join(f"{e:+.3f}" for e in estimates)
          + f"   (spread {max(estimates) - min(estimates):.3f})")
```

*The estimates agree better as the number of hands grows, but slowly: this particular state occurs in only a small fraction of the hands. The error of an average shrinks with the square root of the number of samples, so ten times more hands give about three times less error.*
</details>

### Exercise 4 (No exploration)
Train a second agent for 300,000 hands, but with **no exploration at all**. The simplest way is to copy the function and set `epsilon = 0.0` inside it. Compare its average reward with the agent trained in the notebook, and count how many state-action pairs it tried at least once and how many it tried fewer than 20 times.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def mc_control_greedy(env, episodes, alpha=0.01):
    Q = defaultdict(lambda: np.zeros(2))
    visits = defaultdict(lambda: np.zeros(2))
    for i in range(episodes):
        episode = play_hand(env, lambda state: int(Q[state].argmax()))      # always greedy
        G = 0.0
        for state, action, reward in reversed(episode):
            G = reward + G
            visits[state][action] += 1
            Q[state][action] += alpha * (G - Q[state][action])
    return Q, visits

Q_greedy, visits_greedy = mc_control_greedy(env, 300000)
tried = sum(int((counts > 0).sum()) for counts in visits_greedy.values())
rarely = sum(int(((counts > 0) & (counts < 20)).sum()) for counts in visits_greedy.values())

print(f"greedy from the start: average reward {average_reward(env, lambda s: int(Q_greedy[s].argmax()), 50000):+.3f}")
print(f"with exploration:      average reward {average_reward(env, learned_policy, 50000):+.3f}")
print(f"the greedy agent tried {tried} state-action pairs, {rarely} of them fewer than 20 times")
```

*In this game the greedy agent ends up only a little worse, and it tried almost every pair at least once. Blackjack is forgiving in two ways. The cards put the agent in a random starting state every hand, so it cannot avoid any part of the game. And all estimates start at 0 while most hands lose, so an action that has never been tried looks better than one that has just lost: the accidental optimism of Exercise 3 in Notebook 01. Still, dozens of its choices rest on fewer than 20 tries. In a game that does not shuffle the situations for you, such as the cliff of the next notebook or Snake, an agent that never explores does far worse.*
</details>

### Exercise 5 (Is it better to learn Q or V?)
Use `mc_prediction` to estimate the state values of the **learned** policy (100,000 hands) and draw them with `show_table`. Compare the picture with the one for `stick_on_20` in Section 3. Where has the learned policy improved the value most?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
V_learned = mc_prediction(env, learned_policy, episodes=100000)
show_table(lambda total, dealer, usable: V_learned[(total, dealer, usable)], "Value of the learned policy")

for state in [(16, 10, 0), (13, 5, 0), (18, 7, 0)]:
    print(f"{state}: stick-on-20 policy {V[state]:+.2f}   learned policy {V_learned[state]:+.2f}")
```

*The biggest gains are in the middle totals, 13 to 19. The cautious policy kept hitting there and often went bust. The learned policy stops in time, particularly against a weak dealer card. A state value tells you how good a policy is. Only the action values could tell the agent how to change it.*
</details>

### Exercise 6 (Step sizes (a little harder))
The notebook used a constant step size of 0.01. Train two more agents for 300,000 hands each: one with `alpha = 0.2`, and one that uses the **exact average** (a step of `1 / visits` for each state-action pair, which needs a table of visit counts). Compare the average reward of the three policies. Why is the large step worse? Why is the exact average worse?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def mc_control_average(env, episodes, epsilon_end=0.05):
    Q = defaultdict(lambda: np.zeros(2))
    visits = defaultdict(lambda: np.zeros(2))
    for i in range(episodes):
        epsilon = max(epsilon_end, 1.0 - i / (0.6 * episodes))
        episode = play_hand(env, lambda s: random.randint(0, 1) if random.random() < epsilon else int(Q[s].argmax()))
        G = 0.0
        for state, action, reward in reversed(episode):
            G = reward + G
            visits[state][action] += 1
            Q[state][action] += (G - Q[state][action]) / visits[state][action]
    return Q

Q_large = mc_control(env, 300000, alpha=0.2)
Q_average = mc_control_average(env, 300000)

for label, table in (("alpha = 0.01 (notebook)", Q), ("alpha = 0.2", Q_large), ("exact average", Q_average)):
    score = average_reward(env, lambda s: int(table[s].argmax()), 100000)
    print(f"{label:24s} average reward of the learned policy: {score:+.3f}")
```

*A return in Blackjack is +1, 0 or −1: very noisy. With a step of 0.2 each estimate is dominated by the last few hands, so it keeps jumping around and, where two actions are close, the choice between them flips at random. The exact average has the opposite problem. The game is fixed, but the **policy** is changing: returns collected early were produced by a nearly random player, and the exact average never forgets them. A small constant step sits between the two, and constant step sizes are the rule from the next notebook on.*
</details>

## 🔑 Key takeaways

- A **model-free** agent learns from experience alone: it never sees the transition probabilities.
- **Monte Carlo** estimation averages the returns observed after each state, using complete episodes.
- Without a model the agent must learn **action values** $Q(s, a)$: choosing an action then needs no knowledge of what comes next.
- **Monte Carlo control** interleaves playing with an epsilon-greedy policy and updating the Q-table. Exploration is what keeps every action under review.
- The Blackjack agent rediscovered the outline of **basic strategy** from wins and losses alone.
- Monte Carlo learning waits for the **end of the episode** and its estimates are **noisy**, which motivates the temporal-difference methods of the next notebook.

### 🏁 You have completed Module A!

You now have the vocabulary of the whole field: agents, environments, rewards and returns, policies and value functions, the Bellman equations, and the tension between exploring and exploiting. You have solved a small world exactly by planning, and learned a card game from experience alone.

**Module B** stays with tables but learns faster, one step at a time, and then puts everything to work on a game of our own: Snake.

---
*End of Module A.*

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*